# Master Exam Template: Anomaly / Outlier Detection
## Course Scope: Statistical IQR Method, Z-Score Method, and Isolation Forest
**Matches:** 7CS353 Lab Assignment 4 (Outlier Detection)  
**Marks Rubric (40 Marks):**
* Section 1: EDA [10 Marks]
* Section 2: Implementation (StandardScaler & Baseline Isolation Forest) [10 Marks]
* Section 3: Evaluation (Score distribution, outlier counts, scatter plot) [10 Marks]
* Section 4: Modify Technique (Statistical IQR Method vs Isolation Forest) [10 Marks]

In [ ]:
# 1. Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

plt.rcParams['figure.figsize'] = (8, 4)
print("Libraries imported!")

In [ ]:
# Section 1: EDA (10 Marks) - Quick Exam Dataset Generation
np.random.seed(42)
n_normal = 380
n_anom = 20

# Normal data + Outliers
x1_norm = np.random.normal(50, 8, n_normal)
x2_norm = np.random.normal(100, 15, n_normal)

x1_anom = np.random.uniform(85, 120, n_anom)
x2_anom = np.random.uniform(180, 250, n_anom)

df = pd.DataFrame({
    'Metric_X1': np.concatenate([x1_norm, x1_anom]),
    'Metric_X2': np.concatenate([x2_norm, x2_anom])
}).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Dataset shape:", df.shape)
display(df.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
sns.boxplot(y=df['Metric_X1'], ax=axes[0], color='salmon')
axes[0].set_title("Metric_X1 Outlier Check")
sns.boxplot(y=df['Metric_X2'], ax=axes[1], color='skyblue')
axes[1].set_title("Metric_X2 Outlier Check")
plt.tight_layout()
plt.show()

In [ ]:
# Section 2: Implementation (10 Marks) - Baseline Isolation Forest
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_scaled)

df['is_anomaly_iso'] = np.where(iso.predict(X_scaled) == -1, 1, 0)
df['anomaly_score'] = -iso.decision_function(X_scaled)

print("Isolation Forest fitted!")
print("Anomalies detected:", df['is_anomaly_iso'].sum(), "out of", len(df))

In [ ]:
# Section 3: Evaluation (10 Marks) - Score Distribution & Scatter Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['anomaly_score'], kde=True, ax=axes[0], color='purple')
axes[0].set_title("Distribution of Anomaly Scores")

sns.scatterplot(x='Metric_X1', y='Metric_X2', hue='is_anomaly_iso', data=df, palette={0: 'navy', 1: 'red'}, ax=axes[1], s=40)
axes[1].set_title("Outliers Flagged by Isolation Forest")
plt.tight_layout()
plt.show()

In [ ]:
# Section 4: Modify Technique (10 Marks) - Statistical IQR Method (from Lab 4)
Q1 = df['Metric_X1'].quantile(0.25)
Q3 = df['Metric_X1'].quantile(0.75)
IQR = Q3 - Q1
cutoff = Q3 + 1.5 * IQR

df['is_anomaly_iqr'] = (df['Metric_X1'] > cutoff).astype(int)

comparison_df = pd.DataFrame([
    {'Technique': 'Isolation Forest (Contamination=0.05)', 'Anomalies Flagged': df['is_anomaly_iso'].sum()},
    {'Technique': 'Statistical IQR Method (Lab 4)', 'Anomalies Flagged': df['is_anomaly_iqr'].sum()}
])
print("=== Final Comparison Table ===")
display(comparison_df)